# Peramalan kadar $NO_2$ di daerah Bojonegoro


## Latar Belakang

Peningkatan aktivitas industri, transportasi, serta pertumbuhan populasi yang pesat telah menyebabkan peningkatan signifikan terhadap tingkat pencemaran udara di berbagai wilayah. Salah satu polutan udara utama yang menjadi perhatian adalah Nitrogen Dioksida (NO₂), yaitu gas beracun yang dihasilkan terutama dari proses pembakaran bahan bakar fosil seperti kendaraan bermotor, pembangkit listrik, dan kegiatan industri. NO₂ memiliki dampak serius terhadap kesehatan manusia, seperti gangguan pernapasan, iritasi paru-paru, serta memperburuk penyakit asma dan bronkitis. Selain itu, NO₂ juga berkontribusi terhadap pembentukan hujan asam dan penurunan kualitas lingkungan secara keseluruhan.

## 1. Pengumpulan Data

Pertama kita akan mengumpulkan data Time Series Harian kadar NO2 di daerah Bojonegoro. Pengumpulan data dari sumber website https://dataspace.copernicus.eu/ , buat akun terlebih dahulu di website copernicus tersebut. 

Dokumentasi cara pengambilan data di https://documentation.dataspace.copernicus.eu/notebook-samples/openeo/NO2Covid.html .

Untuk menuliskan code Python untuk mengambil data, silahkan kunjungi halaman https://dataspace.copernicus.eu/analyse/jupyterlab, klik Access JupyterLab, scroll kebawah sedikit ..., lalu pilih Python 3 (ipykernel)

![Teks alternatif](img/Screenshot%202025-10-23%20105545.png)

Disini kita akan mengambil data kadar NO2 di daerah Bojonegoro dari tanggal ... sampai ... .

Kita install terlebih dahulu openoneo:


!pip install openeo



Lalu tuliskan code dibawah:


In [1]:
import openeo


In [3]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()


Authenticated using refresh token.




pada saat menjalankan baris code diatas (connection), nanti akan diminta authentikasi seperti output berikut:


```{code}
Terminal/Output
Visit (link authentikasi) 📋 to authenticate.
✅ Authorized successfully
Authenticated using device code flow.
```




Kalian tinggal klik link authentikasi lalu login menggunakan akun "copernicus" kalian.


In [4]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [111.8596709, -7.1154245],
            [111.6093957, -7.1154245],
            [111.6093957, -7.2272944],
            [111.8596709, -7.2272944],
            [111.8596709, -7.1154245],
        ]
    ]
}

s5post = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2023-10-01", "2025-10-01"],
    spatial_extent={
        "west": 111.6093957,
        "south": -7.2272944,
        "east": 111.8596709,
        "north": -7.1154245
    },
    bands=["NO2"],
)

# Now aggregate by day to avoid having multiple data per day
s5p_no2_daily = s5post.aggregate_temporal_period(reducer="mean", period="day")

# Now create a spatial aggregation to generate mean timeseries data
s5p_no2_aoi = s5p_no2_daily.aggregate_spatial(reducer="mean", geometries=aoi)




Code diatas memerlukan titik koordinasi area yang akan diambil data $NO_2$-nya, untuk mengambil titik koordinasi kaian kunjungi webiste https://geojson.io/#map=12/-7.1713/111.7345 . Didalam website tersebut kalian akan memilih daerah dengan cara memberi shape kotak didaerah yang ingin kalian ambil datanya.

![Teks alternatif](img/Screenshot%202025-10-23%20110952.png)

Di panel sebelah kanan terdapat data JSON yang berupa koordinat daerah yang kalian pilih, kalian salin terus sesuaikan dengan code diatas di bagian variabel "aoi" dan spatial_extent.


Lalu kalian tambahkan baris code dibawah untuk memulai pengambilan data:


In [ ]:
job = s5post.execute_batch(title="NO2 in Bojonegoro", outputfile="NO2Bojonegoro.nc")




Tunggu proses pengambilan data, output proses seperti berikut:


```{code}
0:00:00 Job 'j-2510231608434524a87dedeacfaf5a43': send 'start'
0:00:15 Job 'j-2510231608434524a87dedeacfaf5a43': created (progress 0%)
0:00:20 Job 'j-2510231608434524a87dedeacfaf5a43': created (progress 0%)
0:00:26 Job 'j-2510231608434524a87dedeacfaf5a43': created (progress 0%)
0:00:35 Job 'j-2510231608434524a87dedeacfaf5a43': queued (progress 0%)
0:00:46 Job 'j-2510231608434524a87dedeacfaf5a43': queued (progress 0%)
0:00:58 Job 'j-2510231608434524a87dedeacfaf5a43': queued (progress 0%)
0:01:14 Job 'j-2510231608434524a87dedeacfaf5a43': queued (progress 0%)
0:01:33 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)
0:01:57 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)
0:02:27 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)
0:03:05 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)
0:03:52 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)
0:04:50 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)
0:05:50 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)
0:06:50 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)
0:07:50 Job 'j-2510231608434524a87dedeacfaf5a43': finished (progress 100%)
```




Abaikan ketika ada N/A.

Ketika proses pengambilan data, aktivitas kalian akan terekam di halaman https://editor.openeo.org/?server=https%3A%2F%2Fopeneo.dataspace.copernicus.eu%2Fopeneo%2F1.2 . Disitu terdapat nama dataset dan status pengambilan data.

![Teks alternatif](img/Screenshot%202025-10-24%20121430.png)


## 2. Preproccessing Data

Setelah kita mengambil data, data bisa diunduh di halaman https://editor.openeo.org/?server=https%3A%2F%2Fopeneo.dataspace.copernicus.eu%2Fopeneo%2F1.2 . File akan berbentuk .nc. Kita cuman perlu kolom date dan NO2 menggunakan code dibawah:


In [ ]:
import netCDF4

file_path = "data/NO2Bojonegoro.nc"
ds = netCDF4.Dataset(file_path)

# Lihat seluruh variabel yang tersedia
print("📦 Variabel dalam file:")
print(ds.variables.keys())
# dict_keys(['t', 'x', 'y', 'crs', 'NO2'])

# Ambil NO2
no2 = ds.variables["NO2"][:]

# Ambil Time
time = ds.variables["t"][:]

# Konversi waktu ke format tanggal jika punya atribut 'units'
try:
    time_units = ds.variables["t"].units
    dates = netCDF4.num2date(time, units=time_units)
except Exception:
    dates = time  # fallback kalau tidak ada units

# Tampilkan struktur data NO2
print(type(no2))
# type <class 'numpy.ma.core.MaskedArray'>

print(len(no2))
# banyaknya data record NO2 725

print(len(no2[0]))
# panjang data perbaris 9

print(len(no2[0][0]))
# panjang perdata 8

print(no2[0][0][0])
# 3.7701793e-05




Dari code diatas kita mengetahui bentuk data dari kolom NO2 nya.

jadi struktur data NO2 perbaris adalah:


In [ ]:
[
    [[] * 8] * 9
]




Untuk melihat 10 data pertama adalah:


In [ ]:
print("Contoh data pertama:")
for i in range(0, 10):
    print(no2[i])




Dalam sehari, terdapat banyak data NO2, jadi kita rata-ratakan agar satu cell data hanya terdapat satu value. Namun terdapat masalah pada data NO2 seperti missing value. Contoh pada output dibawah:


```{code}
Terminal/Output
[2.9651806471520104e-05 4.1052295273402706e-05 -- 5.6563803809694946e-05
 -- -- 6.348737952066585e-05 --]
```




### a. Mengatasi Missing Value menggunakan metode Interpolasi Linear

Sekarang kita akan mengatasi permasalahan missing value pada data NO2.


In [ ]:
import numpy as np
import pandas as pd

# Interpolasi Linear
no2_filled = np.zeros_like(no2)
# Untuk jaga-jaga jika terdapat '--' tidak berubah menjadi 0
no2_filled = no2_filled.filled(0)

# loop tiap grid (y,x)
for i in range(no2.shape[1]):     # 9 baris
    for j in range(no2.shape[2]): # 8 kolom
        series = pd.Series(no2[:, i, j])
        no2_filled[:, i, j] = series.interpolate(method='linear', limit_direction='both').to_numpy()




Dengan code diatas, missing value yang terdapat pada data NO2 akan diisi secara otomatis menggunakan metode Interpolasi Linear.

### b. Rata-rata kan Data dan ubah Datetime

Setelah mengatasi missing value, kita akan me-rata-rata-kan data NO2 agar satu record hanya berupa single value. Sekalian kita mengambil date nya dan menaruh di array. Kita akan mengubah datetime dari awalnya (2023-10-04 00:00:00) menjadi (2023-10-04) karena kita mengambil data time series harian jadi kita tidak memerlukan data jam, menit dan detik.


In [ ]:
new_dates = []
new_no2 = []
for i in range(len(dates)):
    # ubah format datetime
    new_date = dates[i].strftime('%Y-%m-%d')
    new_dates.append(new_date)
    new_no2.append(np.mean(no2_filled[i]))




### c. Simpan data dalam bentuk CSV

Setelah itu kita akan membentuk data menjadi DataFrame Pandas untuk disimpan menjadi CSV.


In [ ]:
df = pd.DataFrame({
    "date": dates,
    "NO2": no2_values
})

# Simpan ke CSV
df.to_csv("NO2_Bojonegoro_timeseries.csv", index=False)




Untuk mengatasi missing value dan menyimpan data ke CSV sudah berhasil.


### d. Pengecekan Missing Value data harian pada CSV

Sekarang setelah data berbentuk CSV, kita cek apakah data Time Series harian lengkap. Cara men-cek apakah data Time Series Harian lengkap gunakan code dibawah:


```{code}
import pandas as pd
import numpy as np

df = pd.read_csv("NO2_Bojonegoro_timeseries.csv")

# Pastikan kolom 'date' bertipe datetime
df['date'] = pd.to_datetime(df['date'])

# Buat rentang tanggal lengkap
start_date = "2023-10-01"
end_date = "2025-09-30"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Cek tanggal yang hilang
missing_dates = full_range.difference(df['date'])

print(f"Jumlah hari missing: {len(missing_dates)}")
print("Daftar tanggal missing:")
print(missing_dates)
```


```{code}
output/terminal
Jumlah hari missing: 6
Daftar tanggal missing:
DatetimeIndex(['2023-11-11', '2024-01-01', '2024-03-23', '2024-08-12',
               '2025-01-30', '2025-01-31'],
              dtype='datetime64[ns]', freq=None)
```




Dalam kasus saya ini, terdapat 6 hari missing value. Kita akan mengatasi lagi missing value menggunakan metode Interpolasi Linear. Cara memperbaikinya gunakan code dibawah:


In [ ]:
import pandas as pd

# Pastikan datetime dan sorting
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

# Buat rentang tanggal lengkap
full_range = pd.date_range(start="2023-10-01", end="2025-09-30", freq='D')

# Reindex agar tanggal yang hilang muncul sebagai NaN
df = df.set_index('date').reindex(full_range)
df.index.name = 'date'

# Interpolasi linear berdasarkan indeks waktu
df['NO2'] = df['NO2'].interpolate(method='time')

# (Opsional) jika masih ada NaN di bagian awal/akhir bisa gunakan forward/backward fill
df['NO2'] = df['NO2'].fillna(method='bfill').fillna(method='ffill')

# Simpan kembali ke CSV
df.to_csv("no2_bojonegoro_interpolated.csv")




Setelah saya cek missing value harian, sudah tidak ada lagi missing value.


```{code}
Jumlah hari missing: 0
Daftar tanggal missing:
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
```




dengan bentuk data terdapat 2 kolom, kolom pertama yaitu date atau tanggal, kolom kedua yaitu kadar NO2 yang sudah di rata-rata kan.


```{code}
         date       NO2
0  2023-10-01  0.000027
1  2023-10-02  0.000024
2  2023-10-03  0.000024
3  2023-10-04  0.000021
4  2023-10-05  0.000021
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   date    731 non-null    object
 1   NO2     731 non-null    float64
dtypes: float64(1), object(1)
memory usage: 11.5+ KB
```


## 3. Modeling menggunakan KNN Regression

Dengan data Time Series kadar NO2 harian di daerah Bojonegoro, kita akan memprediksi kadar NO2 satu hari yang akan datang. Sekarang kita akan ubah data, mencoba mencari korelasi antara 1 hari dengan 4 hari sebelumnya. Kita juga akan membandingkan apakah semakin banyak hari sebelumnya, model akan lebih bagus?

### a. Normalisasi Data

karena kita menggunakan model KNN Regression, maka perlu normalisasi data menggunakan min-max Scaler.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

scaler = MinMaxScaler()

df['NO2_scaled'] = scaler.fit_transform(df[['NO2']])




Maka data akan di-normalisasi 0-1.


```{code}
output/terminal
         date       NO2  NO2_scaled
0  2023-10-01  0.000027    0.238203
1  2023-10-02  0.000024    0.192840
2  2023-10-03  0.000024    0.196854
3  2023-10-04  0.000021    0.149560
4  2023-10-05  0.000021    0.154247
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        731 non-null    object
 1   NO2         731 non-null    float64
 2   NO2_scaled  731 non-null    float64
dtypes: float64(2), object(1)
memory usage: 17.3+ KB
```




### b. Mengubah Data

Sekarang saya ingin mengubah data dari sebelumnya hanya 2 fitru menjadi 4 hari sebelum yang terdapat 5 fitur (t-4, t-3, t-2, t-1, dan t sebagai label), saya juga membuat data 10 hari sebelum untuk membandingkan apakah semakin banyak hari sebelum, semakin baik pula modelnya?


In [ ]:
import pandas as pd

def create_supervised(data, n_lag=4):
    df_supervised = pd.DataFrame()
    
    # Membuat fitur t-4 sampai t-1
    for i in range(n_lag, 0, -1):
        df_supervised[f'NO2(t-{i})'] = data.shift(i)
    
    # Label hari H
    df_supervised['NO2(t)'] = data
    
    # Hapus baris yang masih mengandung NaN akibat shift
    df_supervised.dropna(inplace=True)
    
    return df_supervised

# contoh penggunaan
supervised_df = create_supervised(df['NO2_scaled'], n_lag=4)

print(supervised_df)
print(supervised_df.shape)


```{code}
output/terminal
     NO2(t-4)  NO2(t-3)  NO2(t-2)  NO2(t-1)    NO2(t)
4    0.238203  0.192840  0.196854  0.149560  0.154247
5    0.192840  0.196854  0.149560  0.154247  0.185625
6    0.196854  0.149560  0.154247  0.185625  0.152010
7    0.149560  0.154247  0.185625  0.152010  0.149143
8    0.154247  0.185625  0.152010  0.149143  0.159907
..        ...       ...       ...       ...       ...
726  0.123092  0.325742  0.372653  0.145997  0.094458
727  0.325742  0.372653  0.145997  0.094458  0.089599
728  0.372653  0.145997  0.094458  0.089599  0.000000
729  0.145997  0.094458  0.089599  0.000000  0.014405
730  0.094458  0.089599  0.000000  0.014405  0.014405
[727 rows x 5 columns]
(727, 5)
```




Untuk membuat data 10 hari sebelum tinggal tambah code dibawah (ubah parameter n_lag).


In [ ]:
supervised_df10 = create_supervised(df['NO2_scaled'], n_lag=10)
print(supervised_df10)
print(supervised_df10.shape)


```{code}
output/terminal
     NO2(t-10)  NO2(t-9)  NO2(t-8)  NO2(t-7)  NO2(t-6)  NO2(t-5)  NO2(t-4)  NO2(t-3)  NO2(t-2)  NO2(t-1)    NO2(t)
10    0.238203  0.192840  0.196854  0.149560  0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105
11    0.192840  0.196854  0.149560  0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105  0.166780
12    0.196854  0.149560  0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105  0.166780  0.127252
13    0.149560  0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105  0.166780  0.127252  0.083753
14    0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105  0.166780  0.127252  0.083753  0.091532
..         ...       ...       ...       ...       ...       ...       ...       ...       ...       ...       ...
726   0.161874  0.128849  0.095824  0.062799  0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458
727   0.128849  0.095824  0.062799  0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458  0.089599
728   0.095824  0.062799  0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458  0.089599  0.000000
729   0.062799  0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458  0.089599  0.000000  0.014405
730   0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458  0.089599  0.000000  0.014405  0.014405
[721 rows x 11 columns]
(721, 11)
```




### Modeling dan Evaluation

Sekarang dari 2 data yang sudah kita rubah, kita train menggunakan model KNN Regression.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

def train_knn(df_supervised, model_name=""):
    # Pisahkan fitur dan label
    X = df_supervised.drop(columns=['NO2(t)']).values
    y = df_supervised['NO2(t)'].values

    # Split data 80/20
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    # Model KNN Regressor
    knn = KNeighborsRegressor(n_neighbors=5)
    knn.fit(X_train, y_train)

    # Prediksi
    y_pred = knn.predict(X_test)

    # Evaluasi
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print(f"\n=== {model_name} ===")
    print(f"Train Size: {len(X_train)} — Test Size: {len(X_test)}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R² Score: {r2:.4f}")

    return knn, y_test, y_pred


# Train model untuk 4 hari sebelumnya
knn_4, y_test_4, y_pred_4 = train_knn(supervised_df, "KNN - 4 Hari Sebelumnya")

# Train model untuk 10 hari sebelumnya
knn_10, y_test_10, y_pred_10 = train_knn(supervised_df10, "KNN - 10 Hari Sebelumnya")


```{code}
output/terminal
=== KNN - 4 Hari Sebelumnya ===
Train Size: 581 — Test Size: 146
RMSE: 0.065436
R² Score: 0.1395

=== KNN - 10 Hari Sebelumnya ===
Train Size: 576 — Test Size: 145
RMSE: 0.067567
R² Score: 0.0886
```




### Plotting

Plotting untuk visualisasi grafik antara label dan prediksi dari kedua data diatas.

4 hari sebelum:


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure()
plt.plot(np.arange(len(y_test_4)), y_test_4, label="Actual")
plt.plot(np.arange(len(y_pred_4)), y_pred_4, label="Predicted")
plt.title("KNN Regression - 4 Hari Sebelumnya")
plt.xlabel("Sample Index")
plt.ylabel("NO2 Value")
plt.legend()
plt.show()




![Teks alternatif](img/Screenshot%202025-10-25%20163935.png)

10 hari sebelum:


In [ ]:
plt.figure()
plt.plot(np.arange(len(y_test_10)), y_test_10, label="Actual")
plt.plot(np.arange(len(y_pred_10)), y_pred_10, label="Predicted")
plt.title("KNN Regression - 10 Hari Sebelumnya")
plt.xlabel("Sample Index")
plt.ylabel("NO2 Value")
plt.legend()
plt.show()




![Teks alternatif](img/Screenshot%202025-10-25%20164003.png)

Berdasarkan hasil pengujian model KNN Regression terhadap dua skenario jumlah fitur, yaitu 4 hari sebelumnya dan 10 hari sebelumnya, dapat disimpulkan bahwa penambahan jumlah fitur berupa lag harian tidak selalu meningkatkan kinerja model. Pada kasus data konsentrasi NO₂ ini, model dengan 4 hari sebelumnya justru menunjukkan performa yang lebih baik dengan nilai RMSE lebih kecil dan R² yang lebih tinggi dibandingkan model dengan 10 hari sebelumnya. Hal ini mengindikasikan bahwa pola perubahan NO₂ lebih dipengaruhi oleh kondisi dalam beberapa hari terakhir, sehingga penambahan data historis yang terlalu jauh dapat menimbulkan noise dan mengurangi kemampuan model dalam mengenali pola. Selain itu, KNN merupakan metode berbasis kedekatan jarak yang cenderung mengalami penurunan performa ketika jumlah fitur bertambah karena curse of dimensionality. Dengan demikian, penggunaan 4 fitur (4 hari sebelumnya) lebih efektif dalam memodelkan data ini dibandingkan penggunaan 10 fitur.
